<a href="https://colab.research.google.com/github/yogeshwardev/csa6301_Thread_intelligence_network_security/blob/main/lab_programs.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
def caesar_encrypt(text, shift):
    result = ""
    for ch in text:
        if ch.isalpha():
            base = ord('A') if ch.isupper() else ord('a')
            result += chr((ord(ch) - base + shift) % 26 + base)
        else:
            result += ch
    return result


def caesar_decrypt(text, shift):
    return caesar_encrypt(text, -shift)


message = "AttackAtDawn"
shift = 3
enc = caesar_encrypt(message, shift)
dec = caesar_decrypt(enc, shift)
print("Original :", message)
print("Encrypted:", enc)
print("Decrypted:", dec)


Original : AttackAtDawn
Encrypted: DwwdfnDwGdzq
Decrypted: AttackAtDawn


In [3]:
import hashlib


def file_hash(filename, algo="sha256"):
    h = hashlib.new(algo)
    with open(filename, "rb") as f:
        h.update(f.read())
    return h.hexdigest()


with open("sample.txt", "w") as f:
    f.write("Confidential report v1")
print("Original hash :", file_hash("sample.txt"))

with open("sample.txt", "a") as f:
    f.write(" - tampered!")
print("Modified hash :", file_hash("sample.txt"))
print("Integrity check: FAILED (hash changed) -> file was modified")


Original hash : 5a54470e71678d9dbac7ffc51f2cd2a76cc1e8e0d1096bd698eef42229f687d7
Modified hash : 66aa68df2112cb433631ca8675173a88a7e6b2b8a64a4942d01221ee3e4ce2e2
Integrity check: FAILED (hash changed) -> file was modified


In [4]:
import socket


def scan_port(host, port):
    s = socket.socket(socket.AF_INET, socket.SOCK_STREAM)
    s.settimeout(0.5)
    result = s.connect_ex((host, port))
    s.close()
    return result == 0


target = "127.0.0.1"  # scan your own machine (localhost)
ports = [21, 22, 23, 25, 80, 443, 3306, 8080]
print(f"Scanning {target} ...")
for port in ports:
    status = "OPEN" if scan_port(target, port) else "closed"
    print(f"Port {port}: {status}")


Scanning 127.0.0.1 ...
Port 21: closed
Port 22: closed
Port 23: closed
Port 25: closed
Port 80: closed
Port 443: closed
Port 3306: closed
Port 8080: OPEN


In [5]:
import re


def check_url(url):
    reasons = []
    if re.match(r"https?://\d{1,3}\.\d{1,3}\.\d{1,3}\.\d{1,3}", url):
        reasons.append("Uses raw IP address instead of domain name")
    if "@" in url:
        reasons.append("Contains '@' symbol (can hide real destination)")
    if url.count("-") > 3:
        reasons.append("Too many hyphens in domain (common phishing trick)")
    if any(k in url.lower() for k in ["login", "verify", "update", "secure"]) and "https" not in url:
        reasons.append("Suspicious keyword without HTTPS")
    return reasons


urls = [
    "https://www.google.com",
    "http://192.168.10.5/login",
    "http://paypal-verify-account.com@evil.com",
]

for u in urls:
    issues = check_url(u)
    verdict = "SUSPICIOUS" if issues else "Looks OK"
    print(f"\nURL: {u}\nVerdict: {verdict}")
    for r in issues:
        print(" -", r)



URL: https://www.google.com
Verdict: Looks OK

URL: http://192.168.10.5/login
Verdict: SUSPICIOUS
 - Uses raw IP address instead of domain name
 - Suspicious keyword without HTTPS

URL: http://paypal-verify-account.com@evil.com
Verdict: SUSPICIOUS
 - Contains '@' symbol (can hide real destination)
 - Suspicious keyword without HTTPS


In [6]:
import socket

domains = ["www.google.com", "www.python.org", "notarealdomain12345.com"]
for d in domains:
    try:
        ip = socket.gethostbyname(d)
        print(f"{d:30s} -> {ip}")
    except socket.gaierror:
        print(f"{d:30s} -> Could not resolve (invalid/unreachable)")


www.google.com                 -> 142.251.156.119
www.python.org                 -> 151.101.0.223
notarealdomain12345.com        -> Could not resolve (invalid/unreachable)


In [7]:
import socket

domains = ["www.google.com", "www.python.org", "notarealdomain12345.com"]
for d in domains:
    try:
        ip = socket.gethostbyname(d)
        print(f"{d:30s} -> {ip}")
    except socket.gaierror:
        print(f"{d:30s} -> Could not resolve (invalid/unreachable)")


www.google.com                 -> 142.251.156.119
www.python.org                 -> 151.101.0.223
notarealdomain12345.com        -> Could not resolve (invalid/unreachable)


In [8]:
import re
from collections import Counter

# Create the sample log so the script runs standalone (no upload needed)
with open("login_attempts.log", "w") as f:
    f.write("2026-07-08 09:00:01 LOGIN SUCCESS user=alice ip=10.0.0.5\n")
    f.write("2026-07-08 09:01:15 LOGIN FAILED user=admin ip=203.0.113.99\n")
    f.write("2026-07-08 09:01:20 LOGIN FAILED user=admin ip=203.0.113.99\n")
    f.write("2026-07-08 09:01:25 LOGIN FAILED user=admin ip=203.0.113.99\n")
    f.write("2026-07-08 09:01:30 LOGIN FAILED user=admin ip=203.0.113.99\n")
    f.write("2026-07-08 09:01:35 LOGIN FAILED user=admin ip=203.0.113.99\n")
    f.write("2026-07-08 09:02:00 LOGIN SUCCESS user=bob ip=10.0.0.8\n")

failed_by_ip = Counter()
with open("login_attempts.log") as f:
    for line in f:
        if "LOGIN FAILED" in line:
            match = re.search(r"ip=(\S+)", line)
            if match:
                failed_by_ip[match.group(1)] += 1

print("Failed login attempts by IP:")
for ip, count in failed_by_ip.items():
    print(f"  {ip}: {count} attempts")
    if count >= 5:
        print(f"  -> ALERT: possible brute-force attack from {ip}")


Failed login attempts by IP:
  203.0.113.99: 5 attempts
  -> ALERT: possible brute-force attack from 203.0.113.99


In [9]:
import hashlib

# Simulate a stolen password hash (a real system would never expose this!)
stolen_hash = hashlib.sha256("letmein".encode()).hexdigest()

wordlist = ["123456", "password", "admin", "letmein", "qwerty"]

print("Attempting dictionary attack on stolen hash...")
for word in wordlist:
    guess_hash = hashlib.sha256(word.encode()).hexdigest()
    if guess_hash == stolen_hash:
        print(f"Password cracked: '{word}'")
        break
else:
    print("Password not found in wordlist.")


Attempting dictionary attack on stolen hash...
Password cracked: 'letmein'


In [10]:
rules = [
    {"action": "ALLOW", "ip": "10.0.0.5", "port": 443},
    {"action": "ALLOW", "ip": "10.0.0.5", "port": 80},
    {"action": "DENY", "ip": "203.0.113.99", "port": None},  # block this IP entirely
    {"action": "DENY", "ip": None, "port": 23},  # block telnet from anyone
]


def check_packet(ip, port):
    for rule in rules:
        ip_match = rule["ip"] in (None, ip)
        port_match = rule["port"] in (None, port)
        if ip_match and port_match:
            return rule["action"]
    return "DENY"  # default: deny anything not explicitly allowed


packets = [
    ("10.0.0.5", 443),
    ("203.0.113.99", 80),
    ("10.0.0.9", 23),
    ("10.0.0.9", 8080),
]

for ip, port in packets:
    decision = check_packet(ip, port)
    print(f"Packet from {ip}:{port} -> {decision}")


Packet from 10.0.0.5:443 -> ALLOW
Packet from 203.0.113.99:80 -> DENY
Packet from 10.0.0.9:23 -> DENY
Packet from 10.0.0.9:8080 -> DENY


In [11]:
from collections import defaultdict

# Create the sample log so the script runs standalone (no upload needed)
with open("login_attempts.log", "w") as f:
    f.write("2026-07-08 09:00:01 LOGIN SUCCESS user=alice ip=10.0.0.5\n")
    f.write("2026-07-08 09:01:15 LOGIN FAILED user=admin ip=203.0.113.99\n")
    f.write("2026-07-08 09:01:20 LOGIN FAILED user=admin ip=203.0.113.99\n")
    f.write("2026-07-08 09:01:25 LOGIN FAILED user=admin ip=203.0.113.99\n")
    f.write("2026-07-08 09:01:30 LOGIN FAILED user=admin ip=203.0.113.99\n")
    f.write("2026-07-08 09:01:35 LOGIN FAILED user=admin ip=203.0.113.99\n")
    f.write("2026-07-08 09:02:00 LOGIN SUCCESS user=bob ip=10.0.0.8\n")


def simple_ids(logfile, threshold=3):
    attempts = defaultdict(int)
    alerts = []
    with open(logfile) as f:
        for line in f:
            if "LOGIN FAILED" in line:
                ip = line.split("ip=")[1].strip()
                attempts[ip] += 1
                if attempts[ip] == threshold:
                    alerts.append(f"IDS ALERT: {threshold}+ failed logins from {ip}")
    return alerts


for alert in simple_ids("login_attempts.log", threshold=3):
    print(alert)


IDS ALERT: 3+ failed logins from 203.0.113.99


In [12]:
from cryptography.fernet import Fernet

# Install once: pip install cryptography
# Step 1: Generate a shared secret key (both sender & receiver need this)
key = Fernet.generate_key()
cipher = Fernet(key)

# Step 2: Sender encrypts the message before sending over the network
message = b"Transfer $500 to account 12345"
encrypted = cipher.encrypt(message)
print("Encrypted (what an eavesdropper sees):", encrypted)

# Step 3: Receiver decrypts it using the same key
decrypted = cipher.decrypt(encrypted)
print("Decrypted (what the receiver reads):", decrypted.decode())


Encrypted (what an eavesdropper sees): b'gAAAAABqVcsRJeX0Q18xBKdqAgjQ4yaPiTBiDChVIJq5gpcV_E5oFKcKxUlwKCPzAgQfH627abpVqF6vWQU-SrTEJvBf42ok7An94ZtnyUvxOzs3BOUrur8='
Decrypted (what the receiver reads): Transfer $500 to account 12345


In [13]:
import datetime


def generate_report(flagged_ips, source_log):
    lines = []
    lines.append("INCIDENT RESPONSE REPORT")
    lines.append(f"Generated: {datetime.datetime.now()}")
    lines.append(f"Source log: {source_log}")
    lines.append("-" * 40)
    if flagged_ips:
        lines.append(f"{len(flagged_ips)} suspicious IP(s) detected:")
        for ip, count in flagged_ips.items():
            lines.append(f"  - {ip}: {count} failed login attempts")
        lines.append("Recommended action: Block listed IPs, force password reset.")
    else:
        lines.append("No suspicious activity detected.")
    return "\n".join(lines)


flagged = {"203.0.113.99": 5}
report = generate_report(flagged, "login_attempts.log")
print(report)

with open("incident_report.txt", "w") as f:
    f.write(report)


INCIDENT RESPONSE REPORT
Generated: 2026-07-14 05:37:21.048969
Source log: login_attempts.log
----------------------------------------
1 suspicious IP(s) detected:
  - 203.0.113.99: 5 failed login attempts
Recommended action: Block listed IPs, force password reset.


In [14]:
import hashlib


def hash_file(path):
    with open(path, "rb") as f:
        return hashlib.sha256(f.read()).hexdigest()


# Create sample files: one original, one copy, one tampered
with open("original.txt", "w") as f:
    f.write("System32 backup data")
with open("copy.txt", "w") as f:
    f.write("System32 backup data")
with open("tampered.txt", "w") as f:
    f.write("System32 backup data - modified")

files = ["original.txt", "copy.txt", "tampered.txt"]
hashes = {f: hash_file(f) for f in files}
baseline = hashes["original.txt"]
print("Baseline (original.txt):", baseline)

for f in files[1:]:
    status = "IDENTICAL" if hashes[f] == baseline else "MODIFIED/TAMPERED"
    print(f"{f}: {status}")


Baseline (original.txt): 9ac62dc309d11594f1da0d160ded90eca26724461b1c55abeed293c23a0285d1
copy.txt: IDENTICAL
tampered.txt: MODIFIED/TAMPERED
